# LOO Covariate Selection

Run this notebook **once** to select covariates via Leave-One-Covariate-Out.
Results are saved to `outputs/scores/selected_covariates.json` and loaded
automatically by `Bayesian_Global_Model.ipynb`.

You do **not** need to re-run this before every training run.

In [ ]:
import os, sys
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('__file__')), '..'))

os.environ['JAX_PLATFORMS'] = 'cuda'
os.environ['XLA_PYTHON_CLIENT_PREALLOCATE'] = 'false'

import jax
print('JAX devices:', jax.devices())

In [ ]:
import pandas as pd
from utils import add_season_labels, Region_states
from loo   import run_loo
from plotting import plot_loo_heatmap

## Configuration

In [ ]:
FORECAST_SEASON = '2025-2026'

ALL_COVARIATES = (
    ['sin_ew', 'cos_ew']
    + [f'temp_med_lag_{n}'      for n in range(9)]
    + [f'rel_humid_med_lag_{n}' for n in range(9)]
    + [f'precip_med_lag_{n}'    for n in range(9)]
    + ['enso', 'iod', 'pdo', 'biome_encoded']
)
STRUCTURAL_COVARIATES = ['sin_ew', 'cos_ew']

# LOO MCMC settings — reduced for speed, reliable for ranking
LOO_DRAWS  = 500
LOO_TUNE   = 500
LOO_CHAINS = 4

TEST_MODE = False

if TEST_MODE:
    LOO_DRAWS  = 300
    LOO_TUNE   = 300
    LOO_CHAINS = 2
    ACTIVE_STATES     = ['SP', 'AM', 'CE', 'RS', 'GO']
    CANDIDATE_COVARIATES = ['sin_ew', 'cos_ew', 'temp_med_lag_2',
                            'rel_humid_med_lag_3', 'precip_med_lag_6',
                            'enso', 'iod', 'pdo', 'biome_encoded']
else:
    ACTIVE_STATES        = sorted([s for states in Region_states.values() for s in states])
    CANDIDATE_COVARIATES = ALL_COVARIATES

print(f'States ({len(ACTIVE_STATES)}): {ACTIVE_STATES}')
print(f'Candidates ({len(CANDIDATE_COVARIATES)}): {CANDIDATE_COVARIATES}')

## Load Data

In [ ]:
df = pd.read_csv('../data/Hist_dataset.csv.gz', compression='gzip')
df = add_season_labels(df)
df = df[df['uf'].isin(ACTIVE_STATES)].copy()

# Filter to columns present in the data
CANDIDATE_COVARIATES = [c for c in CANDIDATE_COVARIATES if c in df.columns]

print(f'Rows: {len(df):,}')
print(f'Seasons: {sorted(df["season"].unique())}')
print(f'Candidate covariates: {CANDIDATE_COVARIATES}')

## Run LOO

This fits `1 + len(CANDIDATE_COVARIATES) - len(STRUCTURAL_COVARIATES)` models sequentially.
On a GPU with NumPyro this typically takes 1-3 hours for the full 27-state run.

In [ ]:
loo_results, selected_covs, loo_df = run_loo(
    df               = df,
    active_states    = ACTIVE_STATES,
    candidate_covs   = CANDIDATE_COVARIATES,
    structural_covs  = STRUCTURAL_COVARIATES,
    forecast_season  = FORECAST_SEASON,
    loo_draws        = LOO_DRAWS,
    loo_tune         = LOO_TUNE,
    loo_chains       = LOO_CHAINS,
    output_path      = '../outputs/scores/selected_covariates.json',
)

## Results

In [ ]:
print('=== LOO RANKING (most negative = most important) ===')
print(loo_df[['covariate', 'delta_elpd', 'se_delta', 'important']]
      .sort_values('delta_elpd').to_string(index=False))
print(f'\nSelected ({len(selected_covs)}): {selected_covs}')

In [ ]:
plot_loo_heatmap(
    loo_results,
    save_path='../outputs/plots/LOO_Heatmap.png',
)

In [ ]:
loo_df.to_csv('../outputs/scores/loo_results.csv', index=False)
print('Saved: outputs/scores/loo_results.csv')